# MaintIE distillation with a self-hosted open model (vLLM on Colab)

Same pipeline as the DeepSeek version, but the teacher runs on your own Colab GPU with vLLM.
No per-token cost, no rate limits, data stays in Colab.

**Before starting:** *Runtime → Change runtime type → GPU* (A100 or H100). Run `!nvidia-smi` to see which GPU you got.
Check the exact model name on Hugging Face and set `MODEL` below. For a 40 GB GPU, pick a 4-bit (AWQ) version of the model.

## 1. Setup

In [ ]:
!pip install -q -U openai
!git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie 2>/dev/null || echo "already cloned"

## Start the vLLM server (takes a while the first time: install + model download)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
MODEL = "Qwen/Qwen3.6-27B"   # check the exact name on Hugging Face; use a 4-bit AWQ version on a 40 GB GPU
!pip install -q vllm
!pip uninstall -y -q torchaudio   # Colab's torchaudio is built for a different CUDA than vLLM's PyTorch; not needed here
# The server is started (safely) by the next cell

In [ ]:
# (Re)start the vLLM server safely. Run this whenever the server is not running.
import time, requests, subprocess

MODEL = globals().get("MODEL", "Qwen/Qwen3.6-27B")

# 1. Stop the vLLM API server AND its engine process ([v]/[E] stop the pattern matching this command itself)
!pkill -9 -f "[v]llm serve"; pkill -9 -f "[E]ngineCore"

# 2. Wait until the GPU memory is actually free (up to 2 minutes)
for _ in range(24):
    used = int(subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True).stdout.strip().split("\n")[0])
    if used < 2000:
        print(f"GPU free ({used} MiB used)")
        break
    print(f"waiting for GPU to free up... {used} MiB still in use")
    time.sleep(5)
else:
    raise SystemExit("GPU memory is still in use. Restart the runtime and try again.")

# 3. Start exactly one server, in its own session so stopping other cells can't kill it
!setsid nohup vllm serve {MODEL} --max-model-len 32768 --enable-prefix-caching --reasoning-parser qwen3 > /content/vllm.log 2>&1 &

# 4. Wait until it answers, or report a crash
for i in range(240):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=5)
        print("server ready:", r.json()["data"][0]["id"])
        break
    except Exception:
        if subprocess.run(["pgrep", "-f", "[v]llm serve"], capture_output=True).returncode != 0:
            print("vLLM stopped (crashed). Errors from the log:")
            !grep -i -m 10 "error" /content/vllm.log
            break
        if i % 8 == 0:
            print(f"still loading... {i * 15 // 60} min")
        time.sleep(15)


In [ ]:
import json, os, time, random, threading, datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata, drive
from openai import OpenAI

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/mistral_ft/distill"
os.makedirs(OUT_DIR, exist_ok=True)

gold = json.load(open("/content/maintie/data/gold_release.json"))
silver = json.load(open("/content/maintie/data/silver_release.json"))

# Your own vLLM server speaks the same API as OpenAI/DeepSeek
client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1", timeout=1800, max_retries=0)
THINKING = False   # thinking off: ~2 points lower type accuracy on the pilot, many times faster
# MODEL is set in the server cell above (a pinned, downloaded model: results stay reproducible)
print(len(gold), "gold |", len(silver), "silver")

## 2. Allowed types, few-shot examples, duplicates

Allowed detailed types are taken from those used in the gold set (these are the ones your test can evaluate).
`scheme.json` has the full official list; you can switch to it later.

In [ ]:
ENTITY_TYPES = sorted({e["type"] for r in gold for e in r["entities"]})
RELATION_TYPES = sorted({x["type"] for r in gold for x in r["relations"]})
print(len(ENTITY_TYPES), "entity types |", len(RELATION_TYPES), "relation types:", RELATION_TYPES)

# Silver relation names are shortened (e.g. "hasPatient"); map them to the gold names by their last part
REL_BY_LAST = {t.split("/")[-1]: t for t in RELATION_TYPES}
print("unmapped silver relation names:",
      Counter(x["type"] for r in silver for x in r["relations"] if x["type"] not in REL_BY_LAST and x["type"] not in RELATION_TYPES))

# Few-shot examples: fixed gold records. These MUST be excluded from the test set later.
candidates = [i for i, r in enumerate(gold) if len(r["entities"]) >= 3 and len(r["relations"]) >= 2]
random.Random(42).shuffle(candidates)
FEW_SHOT_INDICES = sorted(candidates[:6])
json.dump({"few_shot_indices": FEW_SHOT_INDICES}, open(f"{OUT_DIR}/few_shot_indices.json", "w"))
print("few-shot gold indices:", FEW_SHOT_INDICES)

# Silver texts that also appear in gold: skip them (they would leak into the test set)
gold_texts = {r["text"] for r in gold}
SILVER_TO_LABEL = [i for i, r in enumerate(silver) if r["text"] not in gold_texts]
print("silver to label:", len(SILVER_TO_LABEL), "| skipped duplicates:", len(silver) - len(SILVER_TO_LABEL))

## 3. Prompt

In [ ]:
def render_input(tokens, entities, hint_relations=None):
    # entities: list of dicts with start, end, coarse
    lines = ["Tokens: " + " ".join(f"{i}:{t}" for i, t in enumerate(tokens)), "Entities:"]
    for i, e in enumerate(entities):
        span = " ".join(tokens[e["start"]:e["end"]])
        lines.append(f'{i}: "{span}" | coarse type: {e["coarse"]}')
    if hint_relations:
        lines.append("Existing coarse relations (may be incomplete or wrong):")
        for r in hint_relations:
            lines.append(f'{r["head"]} -> {r["tail"]}: {r["type"]}')
    return "\n".join(lines)

def gold_as_input(rec):
    ents = [{"start": e["start"], "end": e["end"], "coarse": e["type"].split("/")[0]} for e in rec["entities"]]
    return render_input(rec["tokens"], ents)

def silver_as_input(rec):
    ents = [{"start": e["start"], "end": e["end"], "coarse": e["type"].split("/")[0]} for e in rec["entities"]]
    hints = [{"head": x["head"], "tail": x["tail"], "type": REL_BY_LAST.get(x["type"], x["type"])} for x in rec["relations"]]
    return render_input(rec["tokens"], ents, hints)

def gold_as_output(rec):
    return json.dumps({
        "entities": [{"id": i, "type": e["type"]} for i, e in enumerate(rec["entities"])],
        "relations": [{"head": x["head"], "tail": x["tail"], "type": x["type"]} for x in rec["relations"]],
    })

by_class = {}
for t in ENTITY_TYPES:
    by_class.setdefault(t.split("/")[0], []).append(t)

SYSTEM_PROMPT = "\n".join(
    ["You annotate short industrial maintenance work orders using the MaintIE schema.",
     "You receive the tokens and a numbered list of entities with their COARSE type.",
     "Task 1: give each entity the most specific DETAILED type from the allowed list.",
     "The detailed type MUST start with the entity's coarse type. If no finer type fits, use the closest shorter path from the list.",
     "Task 2: list the relations between entities, using entity numbers and the allowed relation types.",
     "Relations point from head to tail. hasPatient: the head (an activity, state or process) affects or describes the tail object.",
     "hasAgent: the tail causes or performs the head. hasPart: the head object has the tail object as a component.",
     "Use only types from the lists below. Do not add or remove entities.",
     'Answer with JSON only, in this form: {"entities": [{"id": 0, "type": "..."}], "relations": [{"head": 0, "tail": 1, "type": "..."}]}',
     "", "ALLOWED ENTITY TYPES:"]
    + [f"[{c}]\n" + "\n".join(ts) for c, ts in sorted(by_class.items())]
    + ["", "ALLOWED RELATION TYPES:"] + RELATION_TYPES + ["", "EXAMPLES:"]
    + [f"INPUT:\n{gold_as_input(gold[i])}\nOUTPUT:\n{gold_as_output(gold[i])}\n" for i in FEW_SHOT_INDICES]
)
print(f"system prompt: {len(SYSTEM_PROMPT):,} characters (about {len(SYSTEM_PROMPT)//4:,} tokens)")
print(silver_as_input(silver[SILVER_TO_LABEL[0]]))

In [ ]:
# Improved prompt: type definitions + example terms + 20 worked examples
scheme = json.load(open("/content/maintie/data/scheme.json"))

def walk(nodes):
    for n in nodes:
        yield n
        yield from walk(n.get("children", []))

scheme_nodes = {n["fullname"]: n for n in walk(scheme["entity"])}
print(len(scheme_nodes), "types in scheme |",
      sum(1 for n in scheme_nodes.values() if n.get("description")), "with descriptions |",
      sum(1 for n in scheme_nodes.values() if n.get("example_terms")), "with example terms")
print("gold types missing from scheme:", [t for t in ENTITY_TYPES if t not in scheme_nodes])

# Prompt pool: 150 gold texts reserved for the prompt (examples and example terms).
# They never go into the test set. The pilot texts are kept out of the pool.
PILOT = json.load(open(f"{OUT_DIR}/pilot_indices.json"))["pilot_indices"]
rest = [i for i in range(len(gold)) if i not in PILOT]
random.Random(123).shuffle(rest)
PROMPT_POOL = sorted(rest[:150])

candidates = [i for i in PROMPT_POOL if len(gold[i]["entities"]) >= 3 and len(gold[i]["relations"]) >= 2]
random.Random(42).shuffle(candidates)
FEW_SHOT_INDICES = sorted(candidates[:20])
json.dump({"prompt_pool_indices": PROMPT_POOL, "few_shot_indices": FEW_SHOT_INDICES},
          open(f"{OUT_DIR}/prompt_pool_indices.json", "w"))
json.dump({"few_shot_indices": FEW_SHOT_INDICES}, open(f"{OUT_DIR}/few_shot_indices.json", "w"))

# Example terms for each type, taken only from the prompt pool
pool_terms = {}
for i in PROMPT_POOL:
    r = gold[i]
    for e in r["entities"]:
        span = " ".join(r["tokens"][e["start"]:e["end"]]).lower()
        pool_terms.setdefault(e["type"], Counter())[span] += 1

def type_line(t):
    node = scheme_nodes.get(t, {})
    line = t
    if node.get("description"):
        line += " - " + node["description"]
    examples = [str(x) for x in node.get("example_terms", [])] + [w for w, _ in pool_terms.get(t, Counter()).most_common(6)]
    examples = list(dict.fromkeys(examples))[:6]
    if examples:
        line += " | e.g. " + ", ".join(examples)
    return line

SYSTEM_PROMPT = "\n".join(
    ["You annotate short industrial maintenance work orders using the MaintIE schema.",
     "You receive the tokens and a numbered list of entities with their COARSE type.",
     "Task 1: give each entity the most specific DETAILED type from the allowed list.",
     "The detailed type MUST start with the entity's coarse type. If no finer type fits, use the closest shorter path from the list.",
     "Physical objects are classified by their FUNCTION (what they do), not by their name.",
     "Task 2: list the relations between entities, using entity numbers and the allowed relation types.",
     "Relations point from head to tail. hasPatient: the head (an activity, state or process) affects or describes the tail object.",
     "hasAgent: the tail causes or performs the head. hasPart: the head object has the tail object as a component.",
     "Use only types from the lists below. Do not add or remove entities. Keep your reasoning brief.",
     'Answer with JSON only, in this form: {"entities": [{"id": 0, "type": "..."}], "relations": [{"head": 0, "tail": 1, "type": "..."}]}',
     "", "ALLOWED ENTITY TYPES (with example terms):"]
    + [f"[{c}]\n" + "\n".join(type_line(t) for t in ts) for c, ts in sorted(by_class.items())]
    + ["", "ALLOWED RELATION TYPES:"] + RELATION_TYPES + ["", "EXAMPLES:"]
    + [f"INPUT:\n{gold_as_input(gold[i])}\nOUTPUT:\n{gold_as_output(gold[i])}\n" for i in FEW_SHOT_INDICES]
)
print(f"few-shot examples: {len(FEW_SHOT_INDICES)} | system prompt about {len(SYSTEM_PROMPT)//4:,} tokens")
print(SYSTEM_PROMPT[:1500])


## 4. Teacher call and validation

In [ ]:
import re

def call_teacher(user_msg, max_tokens=4000):
    for attempt in range(6):
        try:
            resp = client.chat.completions.create(
                model=MODEL, temperature=0, max_tokens=max_tokens,   # room for hidden reasoning + the JSON answer
                extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
                messages=[{"role": "system", "content": SYSTEM_PROMPT},   # identical every time: cached automatically
                          {"role": "user", "content": user_msg}])
            u = resp.usage
            details = getattr(u, "completion_tokens_details", None)
            usage = {"prompt": u.prompt_tokens, "completion": u.completion_tokens,
                     "reasoning": getattr(details, "reasoning_tokens", None) if details else None,
                     "cache_hit": getattr(u, "prompt_cache_hit_tokens", None)}
            choice = resp.choices[0]
            return choice.message.content or "", usage, resp.model, choice.finish_reason
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"retry in {wait:.0f}s: {e}")
            time.sleep(wait)
    raise RuntimeError("teacher failed after retries")

def parse_json(raw):
    # tolerate code fences or extra text around the JSON object
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)   # drop reasoning if it appears in the answer
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None

def validate(pred, coarse):
    # coarse: list of coarse types, one per entity. Returns a list of error strings (empty = passed)
    errors, n = [], len(coarse)
    ents = pred.get("entities") if isinstance(pred, dict) else None
    if not isinstance(ents, list):
        return ["no entity list"]
    if sorted(e.get("id", -1) for e in ents) != list(range(n)):
        errors.append("entity ids do not match input")
    for e in ents:
        t, i = e.get("type"), e.get("id")
        if t not in ENTITY_TYPES:
            errors.append(f"unknown entity type: {t}")
        elif isinstance(i, int) and 0 <= i < n and t.split("/")[0] != coarse[i]:
            errors.append(f"type {t} conflicts with coarse {coarse[i]}")
    for r in pred.get("relations", []):
        h, tl = r.get("head"), r.get("tail")
        if r.get("type") not in RELATION_TYPES:
            errors.append(f"unknown relation type: {r.get('type')}")
        if not (isinstance(h, int) and isinstance(tl, int) and 0 <= h < n and 0 <= tl < n and h != tl):
            errors.append("invalid relation indices")
    return errors

def label(input_text, coarse):
    raw, usage, version, finish = call_teacher(input_text)
    if finish == "length":   # reasoning ran long: retry once with much more room
        raw, usage2, version, finish = call_teacher(input_text, max_tokens=16000)
        usage = {k: (usage.get(k) or 0) + (usage2.get(k) or 0) for k in usage}
    pred = parse_json(raw)
    if pred is None:
        errors = ["cut off at token limit" if finish == "length" else "invalid JSON"]
    else:
        errors = validate(pred, coarse)
    return {"pred": pred, "errors": errors, "raw": raw, "usage": usage, "finish_reason": finish,
            "model_version": version,
            "date": datetime.datetime.now(datetime.timezone.utc).isoformat()}


## 5. Pilot: 50 gold texts (quality check before spending on silver)

Spans are fixed, so this measures: **detailed type accuracy** and **relation F1** against the human labels.

In [ ]:
PILOT_FILE = f"{OUT_DIR}/pilot_indices.json"
if os.path.exists(PILOT_FILE):   # reuse the same 50 texts so runs are comparable
    PILOT = json.load(open(PILOT_FILE))["pilot_indices"]
else:
    pool = [i for i in range(len(gold)) if i not in FEW_SHOT_INDICES]
    PILOT = sorted(random.Random(7).sample(pool, 50))
    json.dump({"pilot_indices": PILOT}, open(PILOT_FILE, "w"))
assert not set(PILOT) & set(FEW_SHOT_INDICES), "pilot overlaps few-shot examples"

def run_one(i):
    rec = gold[i]
    coarse = [e["type"].split("/")[0] for e in rec["entities"]]
    return i, label(gold_as_input(rec), coarse)

# Send all 50 at once: vLLM processes them together on the GPU
start = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    results = dict(ex.map(run_one, PILOT))
print(f"pilot took {(time.time() - start) / 60:.1f} min")
json.dump({str(k): v for k, v in results.items()}, open(f"{OUT_DIR}/pilot_results_selfhosted.json", "w"))

type_ok = type_total = top2_ok = 0
rel_tp = rel_fp = rel_fn = 0
failed, err_counts = 0, Counter()
for i in PILOT:
    rec, res = gold[i], results[i]
    err_counts.update(e.split(":")[0] for e in res["errors"])
    if res["pred"] is None or res["errors"]:
        failed += 1
    pred_types = {e.get("id"): e.get("type") for e in (res["pred"] or {}).get("entities", [])}
    for j, e in enumerate(rec["entities"]):
        type_total += 1
        p = pred_types.get(j) or ""
        type_ok += p == e["type"]
        top2_ok += p.split("/")[:2] == e["type"].split("/")[:2]
    gold_rels = {(x["head"], x["tail"], x["type"]) for x in rec["relations"]}
    pred_rels = {(x.get("head"), x.get("tail"), x.get("type")) for x in (res["pred"] or {}).get("relations", [])}
    rel_tp += len(gold_rels & pred_rels); rel_fp += len(pred_rels - gold_rels); rel_fn += len(gold_rels - pred_rels)

f1 = 2 * rel_tp / max(1, 2 * rel_tp + rel_fp + rel_fn)
reasoning = [r["usage"].get("reasoning") or r["usage"]["completion"] for r in results.values()]
print(f"detailed type accuracy (exact): {type_ok/type_total:.1%}")
print(f"type accuracy (first two levels): {top2_ok/type_total:.1%}")
print(f"relation F1: {f1:.2f}  (P {rel_tp/max(1,rel_tp+rel_fp):.2f}, R {rel_tp/max(1,rel_tp+rel_fn):.2f})")
print(f"texts failing validation: {failed}/50 | error types: {dict(err_counts)}")
print(f"average output tokens per text: {sum(reasoning)/len(reasoning):,.0f} | max: {max(reasoning):,}")


## 6. Label the silver set (resumable)

Results are written to Drive one line per text. If the session stops, re-run this cell: finished texts are skipped.
**Run off-peak.** Check the pilot numbers first.

In [ ]:
import requests
try:
    print("server:", requests.get("http://localhost:8000/v1/models", timeout=5).json()["data"][0]["id"])
except Exception as e:
    raise SystemExit(f"vLLM server is not reachable ({e}). Start it and wait for 'server ready' first.")

OUT_FILE = f"{OUT_DIR}/silver_selfhosted_nothink.jsonl"   # separate file per teacher setting: never mix
done = set()
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        done = {json.loads(line)["idx"] for line in f if line.strip()}
todo = [i for i in SILVER_TO_LABEL if i not in done]
print(f"already done: {len(done)} | to do: {len(todo)}")

lock = threading.Lock()
def work(i):
    rec = silver[i]
    coarse = [e["type"].split("/")[0] for e in rec["entities"]]
    res = label(silver_as_input(rec), coarse)
    res["idx"] = i
    with lock, open(OUT_FILE, "a") as f:
        f.write(json.dumps(res) + "\n")
    return len(res["errors"]) == 0

ok = 0
start = time.time()
with ThreadPoolExecutor(max_workers=48) as pool_ex:
    futures = [pool_ex.submit(work, i) for i in todo]
    for n, fut in enumerate(as_completed(futures), 1):
        ok += fut.result()
        if n % 250 == 0:
            mins = (time.time() - start) / 60
            print(f"{n}/{len(todo)} done, {ok} passed | {mins:.0f} min so far, about {mins / n * (len(todo) - n):.0f} min left")
print("finished")

## 7. Summary: pass rate, errors, tokens and estimated cost

In [ ]:
PRICE_MISS, PRICE_HIT, PRICE_OUT = 0.0, 0.0, 0.0   # self-hosted: no per-token cost

rows = [json.loads(l) for l in open(OUT_FILE) if l.strip()]
passed = [r for r in rows if not r["errors"]]
errs = Counter(e.split(":")[0] for r in rows for e in r["errors"])
prompt = sum(r["usage"]["prompt"] for r in rows)
hit = sum(r["usage"]["cache_hit"] or 0 for r in rows)
out = sum(r["usage"]["completion"] for r in rows)   # includes hidden reasoning tokens
reasoning = sum(r["usage"].get("reasoning") or 0 for r in rows)
cost = (prompt - hit) / 1e6 * PRICE_MISS + hit / 1e6 * PRICE_HIT + out / 1e6 * PRICE_OUT

print(f"labelled: {len(rows)} | passed validation: {len(passed)} ({len(passed)/len(rows):.1%})")
print("error types:", dict(errs))
print(f"tokens: prompt {prompt:,} (cache hits {hit/max(1,prompt):.0%}) | output {out:,} (of which reasoning {reasoning:,})")
print(f"estimated cost (off-peak prices): ${cost:.2f}")
print("model versions seen:", Counter(r["model_version"] for r in rows))